# Calculate Probabilistic Forecast Data (Cleaned)

**Date:** September 3rd, 2025 (Cleaned: October 21st, 2025)  
**Author:** Prakrut Kansara, edited by Kris Su

**Description:** This notebook calculates the `TERCILE CATEGORY PROBABILITY EXCEEDANCE` used for the heatmap figure in the AmazonHydroViewer webapp.

**Improvements in this version:**
- Proper soil profile level handling and saving
- Memory management with garbage collection
- Better path handling with pathlib
- Improved error handling and logging
- Removed duplicate function definitions
- Enhanced documentation

**Surface variable units:**
- `Rainf_tavg`: mm/day
- `Qair_f_tavg`: g/kg
- `Qs_tavg`: mm/day
- `Evap_tavg`: mm/day
- `Tair_f_tavg`: degree Celsius
- `SoilMoist_inst`: m³/m³ (by profile level)
- `SoilTemp_inst`: degree Celsius (by profile level)
- `Streamflow_tavg`: m³/s

In [ ]:
import xarray as xr
import geopandas as gpd
import numpy as np
import re
import gc
from tqdm import tqdm
from datetime import datetime
from pathlib import Path
from shapely.geometry import box

#import os
#import pickle

## Functions

In [ ]:
def read_trim_forecast(file_path, va):
    """
    Read forecast data for a specific variable.
    
    Args:
        file_path (str): Path to forecast NetCDF file
        va (str): Variable name to extract

    Returns:
        xarray.DataArray: Forecast data for the variable
    """
    try:
        forecast_data = xr.open_dataset(file_path)[va]
        return forecast_data
    except KeyError:
        print(f'ERROR: Variable {va} not found in dataset {file_path}')
        raise

In [ ]:
def read_trim_hindcast(file_path, va):
    """
    Read hindcast data for a specific variable from multiple files.
    
    Args:
        file_path (str or list): Path(s) to hindcast NetCDF file(s)
        va (str): Variable name to extract
        
    Returns:
        xarray.DataArray: Hindcast data for the variable, chunked by time
    """
    try:
        hindcast_data = xr.open_mfdataset(file_path)[va].chunk(dict(time=-1))
        return hindcast_data
    except KeyError:
        print(f'ERROR: Variable {va} not found in hindcast dataset')
        raise

In [ ]:
def get_thresh(icat, quantiles, xrds, dims=['ensemble', 'time']):
    """
    Calculate threshold boundaries for a category based on quantiles.
    
    Args:
        icat (int): Category index (0, 1, 2 for terciles)
        quantiles (list): Quantile boundaries (e.g., [1/3, 2/3] for terciles)
        xrds (xarray.DataArray): Data array to calculate quantiles from
        dims (list): Dimensions to calculate quantiles over
    
    Returns:
        tuple: (lower_threshold, upper_threshold) for the category
    """
    if not all(elem in xrds.dims for elem in dims): 
        raise Exception(f'Some dimensions in {dims} not present in xr.DataArray {xrds.dims}')

    if icat == 0:  # Below normal category
        xrds_lo = -np.inf
        xrds_hi = xrds.quantile(quantiles[icat], dim=dims)
    elif icat == len(quantiles):  # Above normal category
        xrds_lo = xrds.quantile(quantiles[icat-1], dim=dims) 
        xrds_hi = np.inf
    else:  # Normal category
        xrds_lo = xrds.quantile(quantiles[icat-1], dim=dims)
        xrds_hi = xrds.quantile(quantiles[icat], dim=dims)
    
    return xrds_lo, xrds_hi

In [ ]:
def calculate_probabilities(hcst, fcst, quantiles=[1/3., 2/3.]):
    """
    Calculate tercile category probability exceedance for ensemble forecast.
    
    Uses hindcast to define climatological tercile boundaries (below-normal, 
    normal, above-normal), then calculates probability that forecast ensemble
    members fall into each category.
    
    Args:
        hcst (xarray.DataArray): Hindcast data with dims [time, ensemble, lat, lon]
        fcst (xarray.DataArray): Forecast data with dims [time, ensemble, lat, lon]
        quantiles (list): Category boundaries (default: terciles at [1/3, 2/3])
    
    Returns:
        xarray.DataArray: Probability (0-1) that forecast falls in each category
                         Dims: [category, time, lat, lon]
                         - Category 0 = below normal (< 33rd percentile)
                         - Category 1 = normal (33rd-67th percentile)
                         - Category 2 = above normal (> 67th percentile)
    """
    print('Computing probabilities...')
    numcategories = len(quantiles) + 1  # 3 categories for terciles

    # Mask out 0 values in forecast (assumes 0 = missing/invalid)
    # NOTE: Verify this is appropriate for your data
    fcst_masked = fcst.where(fcst != 0)

    l_probs = []
    for icat in range(numcategories):
        print(f'  category={icat}')
        h_lo, h_hi = get_thresh(icat, quantiles, hcst)
        
        # Count fraction of ensemble members in this category
        prob = np.logical_and(fcst_masked > h_lo, fcst_masked <= h_hi).sum('ensemble') / float(fcst_masked.sizes['ensemble'])
        
        # Remove quantile coordinate if present (artifact from threshold calculation)
        if 'quantile' in prob.coords:
            prob = prob.drop_vars('quantile')
        
        l_probs.append(prob.assign_coords({'category': icat}))
    
    probs = xr.concat(l_probs, dim='category')
    return probs

In [ ]:
def as_2d(da, xdim="lon", ydim="lat"): 
    """
    Convert multi-dimensional data array to 2D spatial grid.
    
    Drops size-1 dimensions and averages over non-spatial dimensions.
    
    Args:
        da (xarray.DataArray): Input data array
        xdim (str): Longitude dimension name
        ydim (str): Latitude dimension name
        
    Returns:
        xarray.DataArray: 2D array with dims (lat, lon)
    """
    # Drop size-1 dims
    da = da.squeeze(drop=True)
    # Reduce any leftover non-spatial dims (e.g., ensemble/step/time) to mean
    reduce_dims = [d for d in da.dims if d not in (ydim, xdim)]
    if reduce_dims:
        da = da.mean(dim=reduce_dims, skipna=True)
    # Ensure order is (lat, lon)
    return da.transpose(ydim, xdim)

In [ ]:
def extract_river_network_with_attributes(data_array, anomalies_array, flow_threshold):
    """
    Extract river network from gridded streamflow data as polygon cells.
    
    Args:
        data_array (xarray.DataArray): Streamflow values for threshold filtering
        anomalies_array (xarray.DataArray): Anomaly/probability values to store as attributes
        flow_threshold (float): Minimum streamflow to include cell in river network
        
    Returns:
        geopandas.GeoDataFrame: River network with polygon geometries and attributes
    """
    da2d = as_2d(data_array)
    an2d = as_2d(anomalies_array)
    lon = da2d["lon"].values
    lat = da2d["lat"].values
    data = da2d.values
    anom = an2d.values

    # Mask of cells that pass threshold
    mask = np.isfinite(data) & (data >= flow_threshold)
    ii, jj = np.where(mask)

    # Build cell polygons (EPSG:4326 assumes lat/lon grid)
    # Note: Adjust coordinate order if lat is descending
    geoms = [
        box(lon[j], lat[i+1], lon[j+1], lat[i])
        for i, j in zip(ii, jj) if i+1 < len(lat) and j+1 < len(lon)
    ]

    attrs = [
        {"streamflow": float(data[i, j]), "anomaly": float(anom[i, j])}
        for i, j in zip(ii, jj) if i+1 < len(lat) and j+1 < len(lon)
    ]

    return gpd.GeoDataFrame(attrs, geometry=geoms, crs="EPSG:4326")

## Helper Functions for File Management

In [ ]:
# Month name to number mapping
_MONTHS = {m: i+1 for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
)}

# Filename patterns to recognize
_PATTERNS = [
    re.compile(r'^ldas_fcst_(\d{4})_([a-z]{3})(\d{2})\.nc$', re.I),  # ldas_fcst_2024_dec01.nc
    re.compile(r'^ldas_fcst_(\d{4})(\d{2})(\d{2})\.nc$', re.I),      # ldas_fcst_20241201.nc
]

def _parse_date_from_name(name: str) -> datetime | None:
    """
    Parse initialization date from forecast filename.
    
    Supports formats:
    - ldas_fcst_2024_dec01.nc
    - ldas_fcst_20241201.nc
    """
    for pat in _PATTERNS:
        m = pat.match(name)
        if not m:
            continue
        if pat is _PATTERNS[0]:
            y = int(m.group(1))
            mon = _MONTHS.get(m.group(2).lower())
            d = int(m.group(3))
        else:
            y, mon, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if mon and 1 <= mon <= 12 and 1 <= d <= 31:
            return datetime(y, mon, d)
    return None

def forecast_init_datetime(fpath: str) -> datetime:
    """Extract initialization datetime from forecast file path."""
    dt = _parse_date_from_name(Path(fpath).name)
    if dt is None:
        raise ValueError(f"Unrecognized forecast filename format: {fpath}")
    return dt

In [ ]:
def split_forecast_and_dec_hindcasts(
    dir_path: str,
    prefix: str = "ldas_fcst_",
    recursive: bool = False
):
    """
    Find latest forecast file and all December 1st hindcast files.
    
    Args:
        dir_path (str): Directory containing forecast files
        prefix (str): Filename prefix to match
        recursive (bool): Search subdirectories
        
    Returns:
        tuple: (forecast_path, hindcast_paths_list, forecast_datetime)
    """
    base = Path(dir_path)
    if not base.is_dir():
        raise NotADirectoryError(f"Not a directory: {dir_path}")

    pattern = "**/*.nc" if recursive else "*.nc"
    items = []
    for p in base.glob(pattern):
        if not p.is_file():
            continue
        name = p.name
        if not name.startswith(prefix) or not name.endswith(".nc"):
            continue
        dt = _parse_date_from_name(name)
        if dt is None:
            continue
        items.append((dt, p.stat().st_mtime, name, p))

    if not items:
        raise FileNotFoundError(f"No matching .nc files found in {dir_path} (prefix='{prefix}')")

    # Latest by (date, mtime, name)
    items.sort(key=lambda t: (t[0], t[1], t[2]))
    forecast_path = items[-1][3]
    forecast_dt = items[-1][0]

    # Hindcasts = existing Dec-01 files from earlier years only
    hindcasts = [
        p for (dt, _, _, p) in items
        if dt.year < forecast_dt.year and dt.month == 12 and dt.day == 1
    ]
    # Sort hindcasts by year ascending (oldest → newest)
    hindcasts.sort(key=lambda p: _parse_date_from_name(p.name))

    return str(forecast_path), [str(p) for p in hindcasts], forecast_dt

## Configuration & File Discovery

In [ ]:
# Variable definitions
list_of_variables = {
    'Rainf_tavg': 'Average precipitation', 
    'Qair_f_tavg': 'Specific humidity',
    'Qs_tavg': 'Surface runoff',
    'Evap_tavg': 'Evapotranspiration',
    'Tair_f_tavg': 'Avg. air temperature',
    'SoilMoist_inst': 'Soil moisture',
    'SoilTemp_inst': 'Soil temperature',
    'Streamflow_tavg': 'Stream flow'
}

# Data directory
surface_model_file_path = r"/mnt/vast/prakrut/backup/malaria_amazon/amazon_forecast"

# Find forecast and hindcast files
forecast_file, hindcast_files, f_dt = split_forecast_and_dec_hindcasts(surface_model_file_path)

print("Forecast file:", forecast_file)
print("Hindcasts   :", len(hindcast_files), "files")
print("Init date   :", f_dt)

# Create initialization date tag
initialization_date = f"{f_dt.year}_{f_dt.strftime('%b').lower()}"
print("Forecast initialization date:", initialization_date)

## Main Processing Loop

Calculate tercile probabilities for each variable and save results.

For soil variables (`SoilMoist_inst`, `SoilTemp_inst`), data is saved separately by profile level.

In [ ]:
# Create output directory
output_dir = Path('./get_ldas_probabilistics_output')
output_dir.mkdir(exist_ok=True, parents=True)

# Process each variable
for variable, variable_longname in tqdm(list_of_variables.items()):  # Fixed: .items()
    print(f"\n{'='*60}")
    print(f"{variable_longname} ({variable})")
    print('='*60)
    
    try:
        # Load data
        print("Loading hindcast data...")
        hindcast = read_trim_hindcast(hindcast_files, variable)
        print(f"  Shape: {hindcast.shape}")
        
        print("Loading forecast data...")
        forecast = read_trim_forecast(forecast_file, variable)
        print(f"  Shape: {forecast.shape}")
        
        # Calculate probabilities (convert to percentages)
        probs = calculate_probabilities(hindcast, forecast) * 100
        
        # Keep only maximum probability per category
        print("Filtering for maximum probabilities...")
        probs_with_nan = probs.where(probs == probs.max(dim='category'))
        
        # Determine output file path base
        file_base = output_dir / f'prob_{initialization_date}_tercile_probability_max_{variable}'
        
        # Save by profile level for soil variables
        if variable in ['SoilMoist_inst', 'SoilTemp_inst']:
            print(f"\nProcessing soil variable with profile levels...")
            
            # Find profile dimension (various possible names)
            profile_dims = [d for d in probs_with_nan.dims 
                           if 'profile' in d.lower() or d in ['level', 'depth', 'SoilMoist_profiles']]
            
            if profile_dims:
                profile_dim = profile_dims[0]
                n_levels = len(probs_with_nan[profile_dim])
                print(f"  Found {n_levels} levels in dimension: '{profile_dim}'")
                print(f"  Level values: {probs_with_nan[profile_dim].values}")
                
                # Save each level separately
                for level_idx in range(n_levels):
                    level_data = probs_with_nan.isel({profile_dim: level_idx})
                    output_file = f'{file_base}_lvl_{level_idx}.nc'
                    level_data.to_netcdf(output_file)
                    print(f"  ✓ Saved level {level_idx} → {Path(output_file).name}")
            else:
                print(f"  ⚠ WARNING: No profile dimension found")
                print(f"    Available dimensions: {list(probs_with_nan.dims)}")
                print(f"    Saving as single file (lvl_0)")
                probs_with_nan.to_netcdf(f'{file_base}_lvl_0.nc')
        else:
            # Non-soil variables: save as level 0
            output_file = f'{file_base}_lvl_0.nc'
            probs_with_nan.to_netcdf(output_file)
            print(f"  ✓ Saved → {Path(output_file).name}")
        
        print(f"\n✓ Completed {variable}")
        
    except Exception as e:
        print(f"\n✗ ERROR processing {variable}:")
        print(f"  {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    finally:
        # Clean up memory
        print("Cleaning up memory...")
        try:
            del hindcast, forecast, probs, probs_with_nan
        except:
            pass
        gc.collect()

print("\n" + "="*60)
print("✓ All variables processed!")
print("="*60)

## Verification & Visualization

Load saved data and verify output.

In [ ]:
# Example: Load saved streamflow data
streamflow_file = output_dir / f'prob_{initialization_date}_tercile_probability_max_Streamflow_tavg_lvl_0.nc'
if streamflow_file.exists():
    probs_streamflow = xr.open_dataarray(streamflow_file)
    print("Loaded streamflow probabilities:")
    print(probs_streamflow)
else:
    print(f"File not found: {streamflow_file}")

## River Network Extraction (Optional)

Extract river network polygons from hindcast data and probability forecasts.

In [ ]:
# Extract river network from last hindcast time step
# Uncomment to run:

"""data_array = hindcast.isel(time=-1)  # Last time step for spatial structure
streamflow_threshold = 50.0  # m³/s - adjust as needed

fcst_gdf = extract_river_network_with_attributes(
     data_array, 
     probs_streamflow, 
     streamflow_threshold
 )"""

# print(f"Extracted {len(fcst_gdf)} river cells")
# print(fcst_gdf.head())

## Visualization (Optional)

Plot river network with probability anomalies.

In [ ]:
# Uncomment to create multi-category visualization:

# import matplotlib.pyplot as plt
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from matplotlib.colors import BoundaryNorm

# # Define colormap for probabilities
# vmin, vmax = 33.4, 100  # Min tercile probability to max
# n_colors = 10
# cmap = plt.cm.get_cmap('RdYlBu_r', n_colors)  # Red=high prob, Blue=low prob
# bounds = np.linspace(vmin, vmax, n_colors+1)
# norm = BoundaryNorm(boundaries=bounds, ncolors=n_colors)

# # Create figure with 3 subplots (one per category)
# fig, axs = plt.subplots(
#     1, 3,
#     subplot_kw={'projection': ccrs.PlateCarree()},
#     figsize=(18, 6)
# )

# # Amazon basin extent
# lis_output_bounds = [-82, -49, -21, 6]  # [min_lon, max_lon, min_lat, max_lat]

# category_titles = {
#     'cat_0': 'Below Normal (Dry)',
#     'cat_1': 'Normal',
#     'cat_2': 'Above Normal (Wet)'
# }

# # Plot each category
# for idx, (cat_key, ax) in enumerate(zip(['cat_0', 'cat_1', 'cat_2'], axs)):
#     gdf = category_gdfs[cat_key]
    
#     # Set map extent and features
#     ax.set_extent(lis_output_bounds)
#     ax.coastlines(linewidth=0.5)
#     ax.add_feature(cfeature.BORDERS, linewidth=0.3, alpha=0.5)
#     ax.add_feature(cfeature.RIVERS, linewidth=0.3, alpha=0.3)
#     ax.set_facecolor('lightgrey')
    
#     # Plot river network if data exists
#     if len(gdf) > 0:
#         gdf.plot(
#             column='probability',
#             ax=ax,
#             cmap=cmap,
#             norm=norm,
#             linewidth=0,
#             alpha=0.7,
#             legend=False
#         )
#         n_cells = len(gdf)
#         mean_prob = gdf['probability'].mean()
#         ax.set_title(f"{category_titles[cat_key]}\n{n_cells} cells, {mean_prob:.1f}% avg prob", 
#                      fontsize=12, fontweight='bold')
#     else:
#         ax.set_title(f"{category_titles[cat_key]}\nNo data", fontsize=12)
    
#     # Add gridlines
#     gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
#                       linewidth=0.5, alpha=0.3, linestyle='--')
#     gl.top_labels = False
#     gl.right_labels = False
#     if idx > 0:  # Only show y-labels on leftmost plot
#         gl.left_labels = False

# # Add colorbar
# from matplotlib import colorbar
# cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])  # [left, bottom, width, height]
# cbar = colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation='vertical')
# cbar.set_label('Probability (%)', fontsize=12, fontweight='bold')

# # Main title
# fig.suptitle(f'Streamflow Tercile Forecast - {initialization_date.upper()} - Month {time_index+1}',
#              fontsize=16, fontweight='bold', y=0.98)

# plt.tight_layout(rect=[0, 0, 0.91, 0.96])
# plt.savefig(output_dir / f'streamflow_tercile_categories_{initialization_date}_month{time_index}.png',
#             dpi=300, bbox_inches='tight')
# plt.show()